In [ ]:
import mrcfile
import numpy as np

in_file = 'emd_42499.map'
out_file = 'emd_42499_unsharpened.mrc'
sharpening_factor = -79.23  # Negative means unsharpening = blurring

def get_map_params(mrc):
    return {
        "map": [ 
            int(mrc.header.mapc),
            int(mrc.header.mapr),
            int(mrc.header.maps),
        ],  # data organization, 1 for x, 2 for y, 3 for z
        "nbin": [int(mrc.header.nx), int(mrc.header.ny), int(mrc.header.nz)],
        "nbin_tot": int(mrc.header.nx * mrc.header.ny * mrc.header.nz),
        "x0": [
            float(mrc.header.origin.x + float(mrc.header.nxstart) * mrc.voxel_size.x),
            float(mrc.header.origin.y + float(mrc.header.nystart) * mrc.voxel_size.y),
            float(mrc.header.origin.z + float(mrc.header.nzstart) * mrc.voxel_size.z),
        ],
        # dimension of one voxel the in x, y, z directions
        "dx": [
            float(mrc.voxel_size.x),
            float(mrc.voxel_size.y),
            float(mrc.voxel_size.z),
        ],
    }

with mrcfile.open(in_file) as mrc:
    map_params = get_map_params(mrc)
    data = mrc.data.copy()

# Ensure axes correspond to [x, y, z]. This is only required for my mental sanity.
# map_params['map'] has directions corresponding to columns, rows, sections [c, r, s] with 1 = x, 2 = y, 3 = z.
# NumPy convention is row-major order (C-order), so data has shape [s, r, c].
order = np.argsort(map_params['map'][::-1])
data = data.transpose(order)

# Map unsharpening
# Smoothing by a factor σ is a convolution with gaussian g(x) = (σ*sqrt(2π))⁻¹ exp[-x²/(2σ²)]
# Equivalently this is multiplication in Fourier space with G(k) = exp[-2π²k²σ²]
# For B-factors this is G(k) = exp[-Bk²/4] where k is spatial frequency (Å⁻¹) and B has units of Å²
fft_data = np.fft.fftn(data)
freqs = [np.fft.fftfreq(n, d=dx) for n, dx in zip(map_params['nbin'], map_params['dx'])]  # Å⁻¹
kx, ky, kz = np.meshgrid(*freqs, indexing='ij')
k2 = kx**2 + ky**2 + kz**2  # Å⁻²
kernel = np.exp(k2 * sharpening_factor / 4)
fft_data = fft_data * kernel
data = np.real(np.fft.ifftn(fft_data)).astype(np.float32)

data = data.transpose((2, 1, 0))  # Back to [z, y, x]
with mrcfile.new(out_file, overwrite=True) as mrc:
    mrc.set_data(data)
    mrc.voxel_size = map_params["dx"]


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import CenteredNorm

import MDAnalysis as mda

fig, (ax1, ax2) = plt.subplots(ncols=2, sharex=True, sharey=True, figsize=(12, 4))
norm = CenteredNorm()
cmap = 'RdBu_r'
nbin = map_params["nbin"][0]
dx = map_params["dx"][0]
extent = [-0.5*dx, (nbin-0.5)*dx, -0.5*dx, (nbin-0.5)*dx]

with mrcfile.open(in_file) as mrc:
    map_params = get_map_params(mrc)
    data = mrc.data.copy()
# Ensure axes correspond to [x, y, z]
# map_params['map'] has directions corresponding to columns, rows, sections [c, r, s] with 1 = x, 2 = y, 3 = z.
# NumPy convention is row-major order (C-order), so data has shape [s, r, c].
order = np.argsort(map_params['map'][::-1])
data = data.transpose(order)

im = ax1.imshow(data[:, :, len(data)//2].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')

with mrcfile.open(out_file) as mrc:
    map_params = get_map_params(mrc)
    data = mrc.data.copy()
# Ensure axes correspond to [x, y, z]
# map_params['map'] has directions corresponding to columns, rows, sections [c, r, s] with 1 = x, 2 = y, 3 = z.
# NumPy convention is row-major order (C-order), so data has shape [s, r, c].
order = np.argsort(map_params['map'][::-1])
data = data.transpose(order)

ax2.imshow(data[:, :, len(data)//2].T, origin='lower', norm=norm, cmap=cmap, extent=extent, interpolation='bicubic')

# Make sure we did not mess up with the axes by plotting the structure coordinates

u = mda.Universe('9cbu.pdb')
phosphates = u.select_atoms('name P')
mask = (phosphates.positions[:, 2] > L / 2 - 5) & (phosphates.positions[:, 2] < L / 2 + 5)
ax1.scatter(phosphates.positions[mask, 0], phosphates.positions[mask, 1], s=1, color='black')

for ax in (ax1, ax2):
    ax.set_xlabel('x [Å]')
    ax.set_ylabel('y [Å]')
plt.colorbar(im, ax=(ax1, ax2))
plt.xlim(L/4, 3*L/4)
plt.ylim(L/4, 3*L/4)